**Install required RAG libraries**

In [ ]:
!pip install -q chromadb sentence-transformers pypdf ollama

**Import required libraries**

In [ ]:
# Import required libraries

import os
import chromadb
import ollama

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully!")

**Select embedding model**

In [ ]:
# Select free embedding model

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Load embedding model
embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully!")

**Extract text from PDF**

In [ ]:
# Import PDF reader
from pypdf import PdfReader

# Store extracted text
all_pages = []

# Read uploaded PDF
pdf_reader = PdfReader("/content/Muhammad_Fawad_Ali.pdf")

# Read every page
for page_number, page in enumerate(pdf_reader.pages):

    # Extract text from current page
    page_text = page.extract_text() or ""

    # Save page text and page number
    all_pages.append({
        "text": page_text,
        "page": page_number + 1
    })

# Display extracted text
print("Total pages:", len(all_pages))

print("\nExtracted Text:\n")

print(all_pages[0]["text"][:1000])

**Text Chunking**

In [ ]:
# Chunk configuration

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100


# Create chunking function
def split_text(text, chunk_size=500, overlap=100):

    # Remove unnecessary spaces
    text = " ".join(text.split())

    # Store chunks
    chunks = []

    # Starting position
    start = 0

    # Continue until complete text is processed
    while start < len(text):

        # Calculate ending position
        end = start + chunk_size

        # Extract text chunk
        chunk = text[start:end]

        # Save non-empty chunk
        if chunk.strip():
            chunks.append(chunk.strip())

        # Move forward while keeping overlap
        start += chunk_size - overlap

    return chunks


# Create chunks for each PDF page
all_chunks = []

for page in all_pages:

    # Split current page text
    page_chunks = split_text(
        page["text"],
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    # Save each chunk with its page number
    for chunk in page_chunks:

        all_chunks.append({
            "text": chunk,
            "page": page["page"]
        })


# Display results
print("Total chunks:", len(all_chunks))

print("\nFirst chunk:\n")

print(all_chunks[0]["text"])

**Generate Embeddings**

In [1]:
# Extract text from all chunks
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

# Generate embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Convert embeddings into Python lists
embeddings = embeddings.tolist()

# Display embedding information
print("Embeddings generated successfully!")

print("Total embeddings:", len(embeddings))

print("Vector dimensions:", len(embeddings[0]))

NameError: name 'all_chunks' is not defined

**Create ChromaDB Vector Database**

In [ ]:
# Create persistent ChromaDB client
client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

# Create or connect to collection
collection = client.get_or_create_collection(
    name="rag_documents",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB connected successfully!")

**Store Embeddings in ChromaDB**

In [ ]:
# Create unique IDs
ids = [
    f"document_chunk_{i}"
    for i in range(len(all_chunks))
]

# Create metadata
metadatas = [
    {
        "source": "/content/Muhammad_Fawad_Ali.pdf",
        "page": chunk["page"]
    }
    for chunk in all_chunks
]

# Store everything in ChromaDB
collection.add(
    ids=ids,
    documents=chunk_texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print("Documents stored successfully!")

print("Total stored documents:", collection.count())

**Retrieval (Semantic Search)**



In [ ]:
# Create retrieval function
def retrieve_documents(question, top_k=4):

    # Convert question into embedding
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0].tolist()

    # Search similar vectors in ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    # Return matching documents
    return results


# Test retrieval
question = "whats your Technical Skills"

results = retrieve_documents(question)

# Display retrieved chunks
for i, document in enumerate(results["documents"][0]):

    print(f"\nResult {i + 1}")

    print(document)

    print("Source:", results["metadatas"][0][i])

    print("Distance:", results["distances"][0][i])

    print("-" * 50)

**Check Knowledge Base**

In [ ]:
# Check all unique PDF sources
sources = set()

for metadata in collection.get()["metadatas"]:
    sources.add(metadata.get("source"))

print("PDF Sources:", sources)

**Connect Ollama with RAG**




In [ ]:
# Check available Ollama models

import ollama

models = ollama.list()

print(models)

In [ ]:
!ollama pull llama3.2:3b

**AI Answer Generation Function**




In [ ]:
# Generate answer using Ollama

def generate_answer(question, results):

    # Get retrieved documents
    documents = results["documents"][0]

    # Get document metadata
    metadatas = results["metadatas"][0]

    # Combine all retrieved chunks
    context = "\n\n".join(documents)

    # Create prompt for AI
    prompt = f"""
    You are a helpful AI document assistant.

    Answer the question using ONLY the provided context.

    If the answer is not available in the context,
    say that the information was not found
    in the uploaded document.

    Do not invent information.

    DOCUMENT CONTEXT:
    {context}

    USER QUESTION:
    {question}

    YOUR ANSWER:
    """

    # Send prompt to Ollama
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    # Extract AI response
    answer = response["message"]["content"]

    # Return answer and sources
    return answer, metadatas

**Create Gradio Chatbot Function**

In [ ]:
# ==========================================
# 4. GRADIO INTERFACE
# ==========================================

demo = gr.ChatInterface(

    fn=rag_chatbot,

    title="🤖 RAG AI Chatbot",

    description="""
    ### Chat with your PDF Documents

    Ask questions and get AI-generated answers
    based on your knowledge base.
    """,

    examples=[
        "What is the company about?",
        "What products does the company sell?",
        "What is the delivery policy?",
        "What is the return policy?"
    ],

    textbox=gr.Textbox(
        placeholder="Ask your question here...",
        container=True
    )
)


# ==========================================
# 5. LAUNCH APPLICATION
# ==========================================

demo.launch(
    share=False,
    inbrowser=True
)